In [ ]:
setwd("/sc/arion/projects/CommonMind/kyriad02/ICH_Stroke/")
getwd()
args = commandArgs(trailingOnly=TRUE)
print(args)
options(future.globals.maxSize = 100000000 * 1024^2)

# set random seed for reproducibility
set.seed(123456789)
# set.seed(888888888)

print(args)

# ========================= Libraries ===================================

library(viridis)

## We load the required packages
library(dplyr)
library(Seurat)
library(readr)
library(readr)
# single-cell analysis package
# plotting and data science packages
# library(tidyverse)
library(cowplot)
library(patchwork)
library(ggplot2)
# co-expression network analysis packages:
library(igraph)
library(harmony)
library(dittoSeq)
library(RColorBrewer)

library(org.Hs.eg.db)
require(tidyverse)
library(clusterProfiler)
library(WGCNA)
library(hdWGCNA)
library(scCustomize)
library(harmony)


library(SCopeLoomR)
library(dreamlet)
library(SingleCellExperiment)
library(scater)
# library(UCell)
# library(Nebulosa)

library(SingleCellExperiment)
library(scuttle)
library(Seurat)
library(doParallel)
'%notin%' <- Negate('%in%')

# plotting
library(ggplot2)
library(dplyr)
library(RColorBrewer)
library(cowplot)
library(ggtree)
library(aplot)
library(circlize)
library(ComplexHeatmap)

# meta
library(tidyr)
library(muscat)
library(broom)
library(tidyverse)
library(metafor) 

input_rds <- args[1]
group_by <- args[2]
only_microglia <- as.logical(args[3])
n_of_aggregated <- as.numeric(args[4]) # 25
n_of_aggregated_text <- args[4]
threads <- as.numeric(args[5])
output_pdf <- args[6]

workdir <- "/sc/arion/projects/CommonMind/kyriad02/ICH_Stroke/result/Cortex/7.WGCNA/"
source('/sc/arion/work/kyriad02/ICH_project/workflow/scripts/utils.R')
dir.create(workdir)


In [ ]:
#======================= Create Metacells =========================
#======================= Create Metacells =========================
# Read the compressed RDS file containing immune cell data
Immune <- readCRDS('/sc/arion/projects/CommonMind/kyriad02/ICH_Stroke/result/Cortex/4.Transfer_Labels/Immune_Cells.rds.zstd')

# Clean up CellType and CellPop columns
# Replace any occurrences of "Astroccytes" with "Astrocytes" in the CellType column
Immune$CellType[Immune$CellType == "Astroccytes"] <- "Astrocytes"
Immune$CellPop[Immune$CellPop == "Astroccytes"] <- "Astrocytes"

# Replace any occurrences of "Microglia" with "Myeloid" in the CellPop column
Immune$CellPop[Immune$CellPop == "Microglia"] <- "Myeloid"

# Replace any occurrences of "Oligodendrocytes" with "Oligo" in the CellPop column
Immune$CellPop[Immune$CellPop == "Oligodendrocytes"] <- "Oligo"

In [ ]:
# Read the predictions CSV file
predictions_csv <- readr::read_csv('/sc/arion/projects/CommonMind/kyriad02/ICH_Stroke/result/Cortex/5.FreshMG_Mapping/MG_predictions_subtype_ADAM.csv')

# Rename the first column to "Cell" for consistency
colnames(predictions_csv)[1] <- "Cell"

# Set row names of the dataframe to the values in the "Cell" column
rownames(predictions_csv) <- predictions_csv$Cell

# Reorder the predictions dataframe according to the column names of the 'Immune' dataframe
predictions_ordered <- predictions_csv[colnames(Immune),]

# Assign predicted subclass and subtype values to 'Immune' dataframe
Immune$Subclass <- predictions_ordered$predictions_subclass
Immune$Subtype <- predictions_ordered$predictions_subtype

# Replace NA values in 'Subclass' and 'Subtype' with 'CellPop' values
Immune$Subclass[is.na(Immune$Subtype)] <- Immune$CellPop[is.na(Immune$Subtype)]
Immune$Subtype[is.na(Immune$Subtype)] <- Immune$CellPop[is.na(Immune$Subtype)]

# Group similar subtypes under broader subclasses
Immune$Subclass[grep('Adapt',Immune$Subtype)] <- 'Adapt'
Immune$Subclass[grep('ADAM',Immune$Subtype)] <- 'ADAM'
Immune$Subclass[grep('Homeo',Immune$Subtype)] <- 'Homeo'
Immune$Subclass[grep('PVM',Immune$Subtype)] <- 'PVM'
Immune$Subclass[grep('Prolif',Immune$Subtype)] <- 'Prolif'
Immune$Subclass[grep('exAM',Immune$Subtype)] <- 'exAM'

In [ ]:
# Set plot width and height options
options(repr.plot.width = 8, repr.plot.height = 8)

# Generate a dimensional reduction plot (UMAP) of the 'Immune' Seurat object
# using a custom plotting function 'DimPlot_scCustom'
p1 <- DimPlot_scCustom(seurat_object = Immune,   # Seurat object to plot
                       pt.size = 0.05,           # Set point size in the plot
                       group.by = "Subclass",   # Group cells by their subclass
                       reduction = "harmony_umap",  # Use harmony_umap dimensional reduction
                       figure_plot = TRUE) +    # Indicate that the plot is for figure output
      ggtitle("")   # Add an empty title to the plot

# Display the plot
p1



In [ ]:
# Subset the 'Immune' Seurat object to include only cells labeled as 'Myeloid' in the 'CellPop' column
Microglia_only <- subset(Immune, subset = CellPop %in% c('Myeloid'))
# Generate a dimensional reduction plot (UMAP) 
DimPlot(Microglia_only)

In [ ]:
Microglia_only$ICH <- as.numeric(unlist(New_Metadata[Microglia_only$donor,'ICH Score']))
Microglia_only$mRS_binned <- ifelse(Microglia_only$mRS >=4,'B','G') 
Microglia_only$ICH_binned <- ifelse(Microglia_only$ICH >=4,'B','G') 
Microglia_only$Comb2 <- paste0("ICH_",Microglia_only$ICH_binned,'_mRS_',Microglia_only$mRS_binned)
Microglia_only$Comb2 <- factor(Microglia_only$Comb2,levels=c('ICH_G_mRS_G','ICH_G_mRS_B','ICH_B_mRS_B'))

In [ ]:
group_by <- 'Subclass'
DefaultAssay(Microglia_only) <- "RNA"
Microglia_only@reductions$pca <- Microglia_only@reductions$pca_regressed_harmony 
#          Astroccytes          B Cells          exclude        MG-Active
#             1365              623               18            32907
#        MG-Homeo.        MG-Inter.            Murel       Neutrophil
#            21552            25467              813             5805
# Oligodendrocytes    Proliferation          T Cells
#              997             3672             3544
Microglia_only <- SetupForWGCNA(
    Microglia_only,
    gene_select = "fraction", # the gene selection approach
    fraction = 0.05, # fraction of cells that a gene needs to be expressed in order to be included
    wgcna_name = "hdWGCNA_ICH" # the name of the hdWGCNA experiment
)
# construct metacells  in each group
Microglia_only <- MetacellsByGroups(
    seurat_obj = Microglia_only,
    reduction = 'pca_regressed_harmony',
    group.by = c(group_by, "orig.ident"), # specify the columns in seurat_obj@meta.data to group by
    k = 25, # nearest-neighbors parameter
    ident.group = 'Subclass' # set the Idents of the metacell seurat object
)

In [ ]:
# Normalize the gene expression data in the 'Microglia_only' Seurat object
Microglia_only <- NormalizeMetacells(Microglia_only)

# Scale the normalized gene expression data in the 'Microglia_only' Seurat object
# using the variable features identified in the previous step
Microglia_only <- ScaleMetacells(Microglia_only, features = VariableFeatures(Microglia_only))

# Identify variable features in the gene expression data of the 'Microglia_only' Seurat object
Microglia_only <- FindVariableFeatures(object = Microglia_only)

# Perform principal component analysis (PCA) on the variable features of the gene expression data
# in the 'Microglia_only' Seurat object
Microglia_only <- RunPCAMetacells(Microglia_only, features = VariableFeatures(Microglia_only))

set.seed(123456789)
# Perform batch correction using Harmony on the 'Microglia_only' Seurat object
# using the 'orig.ident' as grouping variables
Microglia_only <- RunHarmonyMetacells(Microglia_only, group.by.vars = 'orig.ident',n.seed=123456789)#15678234

# Perform uniform manifold approximation and projection (UMAP) on the 'Microglia_only' Seurat object
# using the Harmony-corrected data and the specified dimensions and parameters
Microglia_only <- RunUMAPMetacells(Microglia_only,
                                    reduction = 'harmony',  # Reduction method
                                    dims = 1:10,            # Dimensions to use
                                    min_dist = 0.5)         # Minimum distance parameter


In [ ]:
# Retrieve the metacell object from 'Microglia_only' and perform neighborhood and clustering identification
All_metacell <- GetMetacellObject(Microglia_only) %>%
        FindNeighbors(reduction = "harmony") %>%
        FindClusters(resolution = seq(0.1, 1, 0.1))  # Cluster at different resolutions

# Assign metadata from 'Microglia_only' to the 'All_metacell' object based on matching 'orig.ident'
All_metacell$Donor <- All_metacell$orig.ident
All_metacell$mRS <- as.vector(Microglia_only$mRS[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$TSH <- as.vector(Microglia_only$TSH[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$race <- as.vector(Microglia_only$race[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$age <- as.vector(Microglia_only$age[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$sex <- as.vector(Microglia_only$sex[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$mRS_binned <- as.vector(Microglia_only$mRS_binned[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$mRS_3class <- as.vector(Microglia_only$mRS_3class[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$BL <- as.vector(Microglia_only$BL[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$CL <- as.vector(Microglia_only$CL[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$dx <- as.vector(Microglia_only$dx[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$PHV <- as.vector(Microglia_only$PHV[match(All_metacell$orig.ident, Microglia_only$orig.ident)])
All_metacell$PEV <- as.vector(Microglia_only$PEV[match(All_metacell$orig.ident, Microglia_only$orig.ident)])


In [ ]:
All_metacell$Clusters <- as.vector(All_metacell$RNA_snn_res.0.4)

All_metacell$Clusters[All_metacell$RNA_snn_res.0.4 %in% c(0,8)] <- 'MTC_1'
All_metacell$Clusters[All_metacell$RNA_snn_res.0.4 %in% c(2)] <- 'MTC_2'
All_metacell$Clusters[All_metacell$RNA_snn_res.0.4 %in% c(5,6)] <- 'MTC_3'
All_metacell$Clusters[All_metacell$RNA_snn_res.0.4 %in% c(3)] <- 'MTC_4'
All_metacell$Clusters[All_metacell$RNA_snn_res.0.4 %in% c(4)] <- 'MTC_5'
All_metacell$Clusters[All_metacell$RNA_snn_res.0.4 %in% c(1)] <- 'MTC_6'
# All_metacell$Clusters[All_metacell$RNA_snn_res.0.4 %in% c(8)] <- 'MTC_7'  
All_metacell$Clusters[All_metacell$RNA_snn_res.0.4==7] <- 'MTC_Prolif'


In [ ]:
options(repr.plot.width=10, repr.plot.height=5)
p<- DimPlot(All_metacell,group.by = 'Subclass',label=F,raster=F,cols = Project_Colors)+
DimPlot(All_metacell,group.by = 'Clusters',label=T,raster=F)+plot_layout(guides = 'collect')

p